# SQL Complete Practice — Part 08: SQL Subqueries
**Topics 132–146 | MySQL | Ankanhaldar**

This notebook covers nested queries (subqueries), their types, operators, and common practical patterns.

> **Execution note:** These are MySQL SQL cells, not Python. Run them in MySQL Workbench, VS Code with SQLTools, or a compatible SQL notebook/kernel. A standard Python kernel does not execute SQL directly.

> **Practice database:** Run the setup cell first. The examples use employee, department, and project tables.

In [ ]:
CREATE DATABASE IF NOT EXISTS ankan_sql_practice;
USE ankan_sql_practice;

CREATE TABLE IF NOT EXISTS subquery_demo_department (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(80) NOT NULL
);

CREATE TABLE IF NOT EXISTS subquery_demo_employee (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    department_id INT NULL,
    salary DECIMAL(10, 2),
    city VARCHAR(50),
    CONSTRAINT fk_subquery_demo_department
        FOREIGN KEY (department_id)
        REFERENCES subquery_demo_department(department_id)
);

INSERT INTO subquery_demo_department (department_id, department_name)
VALUES
    (10, 'IT'),
    (20, 'HR'),
    (30, 'Finance'),
    (40, 'Marketing')
ON DUPLICATE KEY UPDATE department_name = VALUES(department_name);

INSERT INTO subquery_demo_employee
    (employee_id, employee_name, department_id, salary, city)
VALUES
    (1, 'Amit', 10, 60000, 'Kolkata'),
    (2, 'Riya', 20, 45000, 'Howrah'),
    (3, 'Suman', 10, 55000, 'Kolkata'),
    (4, 'Priya', 30, 52000, 'Durgapur'),
    (5, 'Rahul', NULL, 35000, 'Kolkata'),
    (6, 'Neha', 20, 48000, 'Burdwan'),
    (7, 'Arjun', 30, 70000, 'Howrah')
ON DUPLICATE KEY UPDATE
    employee_name = VALUES(employee_name),
    department_id = VALUES(department_id),
    salary = VALUES(salary),
    city = VALUES(city);

CREATE TABLE IF NOT EXISTS subquery_demo_project (
    project_id INT PRIMARY KEY,
    project_name VARCHAR(100) NOT NULL,
    department_id INT
);

INSERT INTO subquery_demo_project (project_id, project_name, department_id)
VALUES
    (101, 'Website Upgrade', 10),
    (102, 'Recruitment Portal', 20),
    (103, 'Budget Automation', 30)
ON DUPLICATE KEY UPDATE
    project_name = VALUES(project_name),
    department_id = VALUES(department_id);

SELECT * FROM subquery_demo_department;
SELECT * FROM subquery_demo_employee;
SELECT * FROM subquery_demo_project;

**Setup note:** The sample inserts refresh the fixed records when rerun. Marketing intentionally has no employees or project, and Rahul has no department, to make unmatched and empty-result cases easier to understand.

## Topic 132: What Is a Subquery?

A **subquery** is a query written inside another SQL statement. The inner query supplies a value or result set that the outer query uses.

A subquery can appear in:
- `WHERE`
- `HAVING`
- `FROM`
- `SELECT`
- `INSERT`, `UPDATE`, or `DELETE` statements (subject to MySQL rules)

Subqueries are useful when one query depends on the result of another query.

### SQL Example — Topic 132

In [ ]:
SELECT employee_name, salary
FROM subquery_demo_employee
WHERE salary > (
    SELECT AVG(salary)
    FROM subquery_demo_employee
);

**Expected result / explanation**

The inner query calculates the average salary. The outer query returns employees whose salary is greater than that average.

## Topic 133: Single-Row Subquery

A **single-row subquery** returns one row. It is commonly used with single-value comparison operators such as `=`, `>`, `<`, `>=`, and `<=`.

The outer query compares a column with the value returned by the inner query. If the subquery unexpectedly returns multiple rows, a scalar comparison can produce an error.

### SQL Example — Topic 133

In [ ]:
SELECT employee_name, salary
FROM subquery_demo_employee
WHERE salary = (
    SELECT MAX(salary)
    FROM subquery_demo_employee
);

**Expected result / explanation**

The inner query returns the highest salary as one value. The outer query returns the employee or employees earning that salary.

## Topic 134: Multiple-Row Subquery

A **multiple-row subquery** returns more than one row. It is commonly used with operators such as `IN`, `ANY`, and `ALL`.

Do not use `=` with a subquery that may return multiple rows. Choose an operator that matches the intended comparison.

### SQL Example — Topic 134

In [ ]:
SELECT employee_name, department_id
FROM subquery_demo_employee
WHERE department_id IN (
    SELECT department_id
    FROM subquery_demo_department
    WHERE department_name IN ('IT', 'Finance')
);

**Expected result / explanation**

The inner query returns the department IDs for IT and Finance. The outer query returns employees belonging to either department.

## Topic 135: Single-Column Subquery

A **single-column subquery** returns one column, even if it returns many rows.

It is often used with `IN`, `NOT IN`, `ANY`, and `ALL`. The number and data types of selected columns must be compatible with the outer comparison.

### SQL Example — Topic 135

In [ ]:
SELECT employee_name, salary
FROM subquery_demo_employee
WHERE department_id IN (
    SELECT department_id
    FROM subquery_demo_department
    WHERE department_name <> 'HR'
);

**Expected result / explanation**

The inner query returns one column: `department_id`. The outer query uses those values to filter employees.

## Topic 136: Multiple-Column Subquery

A **multiple-column subquery** returns more than one column. It can be used with row-constructor comparisons when the outer and inner column counts and types correspond.

This technique is useful when a match depends on a combination of values rather than one column alone.

### SQL Example — Topic 136

In [ ]:
SELECT employee_id, employee_name, department_id, salary
FROM subquery_demo_employee
WHERE (department_id, salary) IN (
    SELECT department_id, MAX(salary)
    FROM subquery_demo_employee
    WHERE department_id IS NOT NULL
    GROUP BY department_id
);

**Expected result / explanation**

The inner query returns each department and its maximum salary. The outer query matches employees on the combined pair of department ID and salary.

## Topic 137: Nested Subquery

A **nested subquery** is a subquery placed inside another subquery. The innermost query runs as part of the dependency chain, and its result is used by the surrounding query.

Nested queries can solve complex problems, but excessive nesting can make SQL difficult to read. Consider a join or a common table expression (CTE) when it improves clarity.

### SQL Example — Topic 137

In [ ]:
SELECT employee_name, salary
FROM subquery_demo_employee
WHERE department_id IN (
    SELECT department_id
    FROM subquery_demo_department
    WHERE department_id IN (
        SELECT department_id
        FROM subquery_demo_project
    )
);

**Expected result / explanation**

The innermost query finds departments with projects. The middle query confirms those department IDs exist in the department table. The outer query returns employees in those departments.

## Topic 138: Correlated Subquery

A **correlated subquery** refers to a column from the outer query. Its result depends on the current outer row.

It is commonly used for comparisons within a group, such as finding employees whose salary is above their own department's average.

A correlated subquery may be evaluated repeatedly or transformed by the optimizer; do not assume a fixed execution strategy.

### SQL Example — Topic 138

In [ ]:
SELECT
    e.employee_name,
    e.department_id,
    e.salary
FROM subquery_demo_employee AS e
WHERE e.department_id IS NOT NULL
  AND e.salary > (
      SELECT AVG(e2.salary)
      FROM subquery_demo_employee AS e2
      WHERE e2.department_id = e.department_id
  );

**Expected result / explanation**

For each employee, the inner query calculates the average salary in that employee's department. The outer query returns employees above their department average.

## Topic 139: Non-Correlated Subquery

A **non-correlated subquery** does not refer to columns from the outer query. It can be evaluated independently of the outer row.

It is often used to calculate one value or a result set that the outer query then uses.

### SQL Example — Topic 139

In [ ]:
SELECT employee_name, salary
FROM subquery_demo_employee
WHERE salary > (
    SELECT AVG(salary)
    FROM subquery_demo_employee
);

**Expected result / explanation**

The inner query calculates one overall average. It does not refer to any outer-query alias, so it is non-correlated.

## Topic 140: Subquery with IN

`IN` checks whether a value matches any value returned by a subquery.

It is useful when the inner query returns a list of values. `NOT IN` can be affected by `NULL` values in the subquery result; if NULLs are possible, consider `NOT EXISTS` instead.

### SQL Example — Topic 140

In [ ]:
SELECT employee_name, department_id
FROM subquery_demo_employee
WHERE department_id IN (
    SELECT department_id
    FROM subquery_demo_department
    WHERE department_name IN ('IT', 'HR')
);

**Expected result / explanation**

The inner query returns the IDs of IT and HR. The outer query returns employees in those departments.

## Topic 141: Subquery with EXISTS

`EXISTS` checks whether a subquery returns at least one row. It is often used with correlated subqueries.

The subquery's selected value is not important; `SELECT 1` is a common convention. `NOT EXISTS` checks that no matching row exists.

### SQL Example — Topic 141

In [ ]:
SELECT
    d.department_id,
    d.department_name
FROM subquery_demo_department AS d
WHERE EXISTS (
    SELECT 1
    FROM subquery_demo_employee AS e
    WHERE e.department_id = d.department_id
);

**Expected result / explanation**

The query returns departments that have at least one employee. Marketing is excluded because no employee references its department ID.

## Topic 142: Subquery with ANY

`ANY` compares an expression with values returned by a subquery. The comparison is true if it succeeds for at least one value.

For example, `salary > ANY (subquery)` means the salary is greater than at least one value returned by the subquery. MySQL also supports `SOME` as a synonym for `ANY`.

### SQL Example — Topic 142

In [ ]:
SELECT employee_name, salary
FROM subquery_demo_employee
WHERE salary > ANY (
    SELECT salary
    FROM subquery_demo_employee
    WHERE department_id = 20
);

**Expected result / explanation**

The query returns employees whose salary is greater than at least one HR employee's salary.

## Topic 143: Subquery with ALL

`ALL` compares an expression with every value returned by a subquery. The comparison must succeed for all values.

For example, `salary > ALL (subquery)` means the salary is greater than every value returned by the inner query.

### SQL Example — Topic 143

In [ ]:
SELECT employee_name, salary
FROM subquery_demo_employee
WHERE salary > ALL (
    SELECT salary
    FROM subquery_demo_employee
    WHERE department_id = 20
);

**Expected result / explanation**

The query returns employees whose salary is greater than every HR employee's salary.

## Topic 144: Subquery in FROM Clause

A subquery in the `FROM` clause is also called a **derived table**. It behaves like a temporary result table for the outer query.

In MySQL, a derived table must have an alias. It is useful for calculating an intermediate result and then querying that result.

### SQL Example — Topic 144

In [ ]:
SELECT
    department_summary.department_id,
    department_summary.average_salary
FROM (
    SELECT
        department_id,
        AVG(salary) AS average_salary
    FROM subquery_demo_employee
    WHERE department_id IS NOT NULL
    GROUP BY department_id
) AS department_summary
WHERE department_summary.average_salary > 50000;

**Expected result / explanation**

The inner query calculates average salary by department. The outer query filters those department-level averages.

## Topic 145: Subquery in SELECT Clause

A subquery can appear in the `SELECT` list as a scalar expression. It must return at most one row and one column.

A correlated scalar subquery can calculate a value for each outer row, such as the employee count for that employee's department.

### SQL Example — Topic 145

In [ ]:
SELECT
    e.employee_name,
    e.department_id,
    (
        SELECT COUNT(*)
        FROM subquery_demo_employee AS e2
        WHERE e2.department_id = e.department_id
    ) AS employees_in_department
FROM subquery_demo_employee AS e;

**Expected result / explanation**

For each employee, the correlated subquery counts employees with the same department ID. For Rahul, whose department ID is NULL, the equality comparison does not match other NULL values, so the count is 0.

## Topic 146: Subquery in INSERT, UPDATE, and DELETE

Subqueries can provide values or conditions for data modification statements.

Common patterns:
- `INSERT ... SELECT` inserts rows returned by a query.
- `UPDATE` with a subquery calculates a value or identifies rows to update.
- `DELETE` with a subquery identifies rows to remove.

These statements change data. Practice them only on test tables and inspect the target rows before running an update or delete.

### SQL Example — Topic 146

In [ ]:
-- INSERT ... SELECT example: create a report table
CREATE TABLE IF NOT EXISTS subquery_demo_high_salary (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    salary DECIMAL(10, 2)
);

INSERT INTO subquery_demo_high_salary (employee_id, employee_name, salary)
SELECT employee_id, employee_name, salary
FROM subquery_demo_employee
WHERE salary > (
    SELECT AVG(salary)
    FROM subquery_demo_employee
)
ON DUPLICATE KEY UPDATE
    employee_name = VALUES(employee_name),
    salary = VALUES(salary);

SELECT * FROM subquery_demo_high_salary;

-- UPDATE example (commented out; run only if you intend to modify data):
-- UPDATE subquery_demo_employee
-- SET salary = salary * 1.05
-- WHERE department_id IN (
--     SELECT department_id
--     FROM subquery_demo_department
--     WHERE department_name = 'HR'
-- );

-- DELETE example (commented out; destructive):
-- DELETE FROM subquery_demo_high_salary
-- WHERE salary < 50000;

**Expected result / explanation**

The active `INSERT ... SELECT` copies employees earning above the overall average into a report table. The update and delete examples are commented out so they cannot modify data accidentally.

## Quick Revision Table

| Subquery type / placement | Meaning |
|---|---|
| Single-row | Returns one row |
| Multiple-row | Returns more than one row |
| Single-column | Returns one column |
| Multiple-column | Returns more than one column |
| Nested | Subquery inside another subquery |
| Correlated | Refers to the outer query |
| Non-correlated | Independent of the outer query |
| `IN` | Matches one of the returned values |
| `EXISTS` | Tests whether at least one row exists |
| `ANY` | Comparison succeeds for at least one value |
| `ALL` | Comparison succeeds for every value |
| `FROM` subquery | Derived table |
| `SELECT` subquery | Scalar expression, at most one row |
| DML subquery | Used in INSERT, UPDATE, or DELETE |

## Important Differences

### Correlated vs Non-Correlated
- Correlated subquery refers to the outer query.
- Non-correlated subquery runs independently of the outer query.

### IN vs EXISTS
- `IN` compares a value with a set of returned values.
- `EXISTS` checks whether at least one matching row exists.

### ANY vs ALL
- `> ANY` means greater than at least one value.
- `> ALL` means greater than every value.

### NOT IN vs NOT EXISTS
- `NOT IN` can behave unexpectedly if the subquery returns `NULL`.
- `NOT EXISTS` is often safer for anti-join logic when NULLs may occur.

## Practice Checklist

- [ ] Find employees earning above the overall average.
- [ ] Find the employee with the highest salary.
- [ ] Use a multiple-row subquery with IN.
- [ ] Write a multiple-column subquery.
- [ ] Write a nested subquery.
- [ ] Find employees above their department average using a correlated subquery.
- [ ] Identify departments that have employees using EXISTS.
- [ ] Compare ANY and ALL.
- [ ] Use a derived table in the FROM clause.
- [ ] Use a scalar subquery in the SELECT clause.
- [ ] Practice INSERT ... SELECT safely.

**End of Part 08 — Topics 132–146.**